---
numbering: false
---

# GEE Time Series

Extract time-series of an NDI (by default, NDVI) within a small, homogenous sample region from the complete Landsat 8/9 record using Google Earth Engine (GEE). Three different time-series are produced and saved as CSV files:

 - **Mean monthly NDI**: average January-December NDI values across a number of years (e.g. January 2014-25, February 2014-25, `...`, December 2014-25).
 - **Time-series mean monthly NDI**: average NDI values for every month between two years (e.g. January 2020, February 2020, `...`, December 2023).
 - **Time-series mean seasonal NDI**: average NDI values over a set period, every year (e.g. August-October 2014, August-October 2015, `...`, August-October 2025).

Each section is independent: you only need to run the ones you want.

In [ ]:
import ee      # The main Earth Engine library
import geemap  # A Python package for interactive mapping with Google Earth Engine

import pandas as pd

import matplotlib.pyplot as plt

Connect to Google Earth Engine. You will need to provide your own project ID (if in doubt, go to [code.earthengine.google.com](https://code.earthengine.google.com/) and see the name in the top-right corner).

In [ ]:
# =====================================
# Replace this with your own project id
my_project_name = "ee-trchudley"
# =====================================

ee.Authenticate()
ee.Initialize(project=my_project_name)

## Set parameters

Set the sample region and NDI. The sample region should represent a single, homogenous surface type (e.g. entirely forest). Note we use Top of Atmosphere (TOA) data here, as the cloud masking below requires it, so bands are named e.g. `B4` rather than `SR_B4`.

In [ ]:
# Location - editable
latitude = 41.017          # Degrees of latitude
longitude = -123.611       # Degrees of longitude
sample_radius = 1000       # radius of sample area around chosen point, in metres
location_name = "hoopa"    # recognisable name, to create a useful file name

# NDI - editable
type_of_ndi = "NDVI"  # type of NDI calculated, for column and filename purposes
r_high = "B5"         # relevant band name for r_high (Landsat 8/9 TOA)
r_low = "B4"          # relevant band name for r_low (Landsat 8/9 TOA)

# Cloud filtering - editable
max_cloud_percent = 70     # maximum scene cloud cover percentage
max_cloud_score = 20       # maximum per-pixel cloud score (0-100) - pixels above this are masked

# Set up location geometry
point = ee.Geometry.Point(longitude, latitude)  # Create a point
sample_region = point.buffer(sample_radius)     # Buffer the point by the sample radius to create a circular region

Visualise the sample region. To check you are sampling a consistent surface, you can bring up a satellite basemap by clicking 🔧 > 🗺️ and selecting the SATELLITE provider.

In [ ]:
map = geemap.Map()
map.addLayer(sample_region, {}, "Sample Region")
map.centerObject(sample_region, zoom=12)
map

## Get the cloud-masked Landsat record

Get the complete Landsat 8/9 TOA record over the sample region, filtered by scene cloud cover, and with cloudy pixels masked using GEE's `simpleCloudScore` algorithm.

In [ ]:
# Get Landsat 8 and 9 image collections, merged together
landsat_collection = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA"))

# Filter to the sample region and scene cloud cover, and sort by date
landsat_collection = (
    landsat_collection
    .filterBounds(sample_region)
    .filter(ee.Filter.lt("CLOUD_COVER", max_cloud_percent))
    .sort("system:time_start")
)

# Mask cloudy pixels within every image
landsat_collection = landsat_collection.map(
    lambda image: image.updateMask(ee.Algorithms.Landsat.simpleCloudScore(image).select("cloud").lte(max_cloud_score))
)

Define a function that summarises the NDI of a collection of images within the sample region. Each value is the mean (and standard deviation) NDI within the sample region of a median NDI composite of all cloud-masked images in that period. Periods with no images return blank (`NaN`) values.

In [ ]:
def calc_ndi_stats(collection):
    """Calculate the mean and standard deviation NDI within the sample region of a median NDI composite"""

    # If there are no available images, return None (blank)
    if collection.size().getInfo() == 0:
        return None, None

    # Calculate NDI for every image in the collection, and produce a median composite
    ndi_composite = collection.map(lambda image: image.normalizedDifference([r_high, r_low]).rename(type_of_ndi)).median()

    # Calculate the mean and standard deviation of the composite within the sample region
    stats = ndi_composite.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=sample_region,
        scale=30,
    ).getInfo()

    return stats[f"{type_of_ndi}_mean"], stats[f"{type_of_ndi}_stdDev"]

## Mean monthly NDI

Average NDI for each calendar month across a range of years. Landsat 8 launched in February 2013, so full years are available from 2014 onwards.

In [ ]:
year_begin = 2014
year_end = 2025

In [ ]:
records = []

# Filter record to years
filtered_collection = landsat_collection.filter(ee.Filter.calendarRange(year_begin, year_end, "year"))

# Simple 'loading bar' showing the months
print("Month:  JFMAMJJASOND\n        ", end="")

for month in range(1, 13):
    print(".", end="")

    # Filter images only to those within the chosen month, and calculate NDI statistics
    month_images = filtered_collection.filter(ee.Filter.calendarRange(month, month, "month"))
    mean_ndi, stddev_ndi = calc_ndi_stats(month_images)

    records.append({"month": month, type_of_ndi: mean_ndi, "stddev": stddev_ndi})

monthly_ndi_record = pd.DataFrame(records)

print("\n        Finished!")

monthly_ndi_record

In [ ]:
fig, ax = plt.subplots(layout="constrained")

# plot the NDI record
monthly_ndi_record.plot('month', type_of_ndi, ax=ax)

# plot the standard deviation as a shaded area around the mean
ax.fill_between(
    monthly_ndi_record['month'],
    monthly_ndi_record[type_of_ndi] - monthly_ndi_record["stddev"],
    monthly_ndi_record[type_of_ndi] + monthly_ndi_record["stddev"],
    color="gray", alpha=0.2, label="Standard Deviation",
)

ax.set_xlabel('Month')
ax.set_ylabel(type_of_ndi)
ax.legend()

plt.show()

In [ ]:
filename = f"{location_name}_{type_of_ndi}_mean_monthly_{year_begin}-{year_end}.csv"

print("Saving to:", filename)

monthly_ndi_record.to_csv(filename, index=False)

## Time-series mean monthly NDI

Average NDI for every individual month between two years (inclusive). This requires more requests to GEE, so takes longer to run over long time periods.

In [ ]:
year_begin = 2020
year_end = 2023

In [ ]:
records = []

# Simple 'loading bar'
print("Year    JFMAMJJASOND")

for year in range(year_begin, year_end + 1):
    print(f"{year}    ", end="")

    for month in range(1, 13):
        print(".", end="")

        # Filter images only to those within the chosen year and month, and calculate NDI statistics
        month_images = (
            landsat_collection
            .filter(ee.Filter.calendarRange(year, year, "year"))
            .filter(ee.Filter.calendarRange(month, month, "month"))
        )
        mean_ndi, stddev_ndi = calc_ndi_stats(month_images)

        # Assign a nominal date of the 15th of the month (but remember this represents a monthly average)
        records.append({"date": pd.Timestamp(year=year, month=month, day=15), type_of_ndi: mean_ndi, "stddev": stddev_ndi})
    print("")

timeseries_ndi_record = pd.DataFrame(records)

print("Finished!")

timeseries_ndi_record

In [ ]:
fig, ax = plt.subplots(layout="constrained")

# plot the NDI record
timeseries_ndi_record.plot('date', type_of_ndi, ax=ax)

# plot the standard deviation as a shaded area around the mean
ax.fill_between(
    timeseries_ndi_record['date'],
    timeseries_ndi_record[type_of_ndi] - timeseries_ndi_record["stddev"],
    timeseries_ndi_record[type_of_ndi] + timeseries_ndi_record["stddev"],
    color="gray", alpha=0.2, label="Standard Deviation",
)

ax.set_xlabel('Date')
ax.set_ylabel(type_of_ndi)
ax.legend()

plt.show()

In [ ]:
filename = f"{location_name}_{type_of_ndi}_timeseries_monthly_{year_begin}-{year_end}.csv"

print("Saving to:", filename)

timeseries_ndi_record.to_csv(filename, index=False)

## Time-series mean seasonal NDI

A single average NDI value per year, calculated over a chosen range of months (inclusive) - e.g. 8 and 10 for August-October, or 1 and 12 for the whole year.

In [ ]:
year_begin = 2014
year_end = 2025

month_begin = 8
month_end = 10

In [ ]:
records = []

# Simple 'loading bar'
print("Year")

for year in range(year_begin, year_end + 1):
    print(f"{year}... ", end="")

    # Filter images only to those within the chosen year and months, and calculate NDI statistics
    season_images = (
        landsat_collection
        .filter(ee.Filter.calendarRange(year, year, "year"))
        .filter(ee.Filter.calendarRange(month_begin, month_end, "month"))
    )
    mean_ndi, stddev_ndi = calc_ndi_stats(season_images)

    records.append({"year": year, type_of_ndi: mean_ndi, "stddev": stddev_ndi})

seasonal_ndi_record = pd.DataFrame(records)

print("Finished!")

seasonal_ndi_record

In [ ]:
fig, ax = plt.subplots(layout="constrained")

# plot the NDI record
seasonal_ndi_record.plot('year', type_of_ndi, ax=ax)

# plot the standard deviation as a shaded area around the mean
ax.fill_between(
    seasonal_ndi_record['year'],
    seasonal_ndi_record[type_of_ndi] - seasonal_ndi_record["stddev"],
    seasonal_ndi_record[type_of_ndi] + seasonal_ndi_record["stddev"],
    color="gray", alpha=0.2, label="Standard Deviation",
)

ax.set_xlabel('Year')
ax.set_ylabel(type_of_ndi)
ax.legend()

plt.show()

In [ ]:
filename = f"{location_name}_{type_of_ndi}_timeseries_seasonal_months-{month_begin:02d}-{month_end:02d}_{year_begin}-{year_end}.csv"

print("Saving to:", filename)

seasonal_ndi_record.to_csv(filename, index=False)